# Week 2-3: Feature Extraction Pipeline

This notebook extracts MFCC, LFCC, CQCC (cached), and Spectral features from audio files.

**Smoke Test Mode**: If `configs/features.yaml` has `smoke_test: true`, reads from `data/smoke_test/` instead of full training set and runs all feature extraction in <5 minutes.

**Validates**:
- DSP cache produces correct shapes
- Utterance stats compute without NaN
- HDF5 chunking round-trips correctly
- `StandardScaler` + model `fit()` works
- SHAP `TreeExplainer` accepts model

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Verify Drive paths
import os
base = "/content/drive/MyDrive/DeepFakeVoiceResearch"
required = [
    "datasets/ASVspoof2019_LA",
    "datasets/In_The_Wild",
    "datasets/smoke_test/flac",
    "features",
    "models/mlruns",
    "results",
    "figures"
]
for f in required:
    p = os.path.join(base, f)
    if not os.path.exists(p):
        os.makedirs(p, exist_ok=True)
        print(f"[CREATED] {p}")
    else:
        print(f"[OK] {p}")
print("\nDrive ready ✅")

ModuleNotFoundError: No module named 'h5py'

In [ ]:
!pip install -q librosa soundfile scipy numpy pandas pyyaml h5py
!pip install -q xxhash  # for fast cache hashing

In [ ]:
%cd /content/drive/MyDrive/DeepFakeVoiceResearch

In [ ]:
!ls

In [ ]:
import yaml

# Load the current config
with open('configs/features.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Change the setting to True
config['smoke_test'] = True

# Save it back to the file
with open('configs/features.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Successfully updated features.yaml to smoke_test: True")

In [ ]:
!cat configs/features.yaml
print("----------------------")
!cat configs/paths.yaml

In [ ]:
import os
import shutil
import pandas as pd
import yaml

# Load paths from your config
with open('configs/paths.yaml') as p:
    paths = yaml.safe_load(p)

# 1. Set up source and destination paths
asv_root = paths['asvspoof_2019']['root']
train_audio_dir = os.path.join(asv_root, paths['asvspoof_2019']['train_audio'])
protocol_path = os.path.join(asv_root, paths['asvspoof_2019']['protocols'], paths['asvspoof_2019']['train_protocol_file'])

smoke_root = paths['smoke_test']['root']
bonafide_dir = os.path.join(smoke_root, 'bonafide')
spoof_dir = os.path.join(smoke_root, 'spoof')

# 2. Read the labels from the protocol file
print("Reading protocol file...")
df = pd.read_csv(protocol_path, sep=" ", header=None, names=["speaker", "filename", "system", "null", "label"])

# 3. Randomly sample 50 files for each class
bonafide_files = df[df['label'] == 'bonafide'].sample(n=50, random_state=42)['filename'].tolist()
spoof_files = df[df['label'] == 'spoof'].sample(n=50, random_state=42)['filename'].tolist()

# 4. Copy the files
def copy_files(file_list, dest_dir):
    copied = 0
    for f in file_list:
        # ASVspoof files end in .flac
        src = os.path.join(train_audio_dir, f"{f}.flac")
        dst = os.path.join(dest_dir, f"{f}.flac")

        if os.path.exists(src):
            shutil.copy2(src, dst)
            copied += 1
        else:
            print(f"Warning: Could not find {src}")
    return copied

print("Copying 50 bonafide files...")
b_count = copy_files(bonafide_files, bonafide_dir)

print("Copying 50 spoof files...")
s_count = copy_files(spoof_files, spoof_dir)

print(f"\n✅ Done! Successfully copied {b_count} bonafide and {s_count} spoof files.")

In [ ]:
import yaml
import os
import glob

# Load BOTH configs
with open('configs/features.yaml') as f:
    feat_config = yaml.safe_load(f)
with open('configs/paths.yaml') as p:
    path_config = yaml.safe_load(p)

# Use the new corrected key
SMOKE_TEST = feat_config.get('smoke_test_mode', False)
print(f"Smoke test mode: {SMOKE_TEST}")

if SMOKE_TEST:
    # Read the path from paths.yaml instead
    SMOKE_ROOT = path_config['smoke_test']['root']
    print(f"Smoke path: {SMOKE_ROOT}")

    # Ensure the subdirectories exist
    bonafide_dir = os.path.join(SMOKE_ROOT, 'bonafide')
    spoof_dir = os.path.join(SMOKE_ROOT, 'spoof')
    os.makedirs(bonafide_dir, exist_ok=True)
    os.makedirs(spoof_dir, exist_ok=True)

    bonafide = sorted(glob.glob(os.path.join(bonafide_dir, '*.flac')))
    spoof = sorted(glob.glob(os.path.join(spoof_dir, '*.flac')))

    print(f"Bonafide files: {len(bonafide)}")
    print(f"Spoof files: {len(spoof)}")
    print(f"Total: {len(bonafide) + len(spoof)}")

In [ ]:
import sys
import os
# Force Colab to look in your project folder for the 'src' modules
PROJECT_ROOT = '/content/drive/MyDrive/DeepFakeVoiceResearch'
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import yaml
import os
import h5py
from pathlib import Path

# 1. Load BOTH configurations
with open('configs/features.yaml') as f:
    feat_config = yaml.safe_load(f)
with open('configs/paths.yaml') as p:
    path_config = yaml.safe_load(p)

# 2. Safely read the smoke test boolean (using the fixed key name)
SMOKE_TEST = feat_config.get('smoke_test_mode', False)
print(f'Smoke test mode: {SMOKE_TEST}')

# 3. Set paths based on paths.yaml
if SMOKE_TEST:
    DATA_ROOT = path_config['smoke_test']['root']
    print(f'Using smoke test data: {DATA_ROOT}')
else:
    DATA_ROOT = path_config['asvspoof_2019']['root']
    print(f'Using full data: {DATA_ROOT}')

# 4. Create output directories
FEATURES_DIR = path_config['output']['features']
os.makedirs(FEATURES_DIR, exist_ok=True)
print(f'Features will be saved to: {FEATURES_DIR}')

In [ ]:
import os
print("1. Does 'src' exist? ", os.path.exists('src'))
print("2. Does 'src/utils' exist? ", os.path.exists('src/utils'))
print("3. Does 'cache_loader.py' exist? ", os.path.exists('src/utils/cache_loader.py'))

In [ ]:
import os

# Ensure the directory exists
os.makedirs('src/utils', exist_ok=True)

# Write the cache_loader.py code
cache_loader_code = """import h5py
import numpy as np
from pathlib import Path

def load_cached_cqcc(h5_path, file_names, n_cqcc=20, max_frames=300):
    \"\"\"
    Loads precomputed CQCC features from an HDF5 cache file on Google Drive.
    Returns a dictionary mapping file names to their corresponding CQCC numpy arrays.
    \"\"\"
    cache_dict = {}
    
    if not Path(h5_path).exists():
        print(f"[WARNING] Cache file not found at: {h5_path}")
        return cache_dict

    with h5py.File(h5_path, 'r') as hf:
        if "cqcc" not in hf or "file_names" not in hf:
            print(f"[ERROR] Invalid cache structure in {h5_path}")
            return cache_dict
            
        stored_files = [f.decode('utf-8') if isinstance(f, bytes) else str(f) for f in hf["file_names"][:]]
        dset = hf["cqcc"]
        
        # If file_names list is empty, load everything (used for smoke tests)
        if not file_names:
            for i, fname in enumerate(stored_files):
                cache_dict[fname] = dset[i]
        else:
            # Create a fast lookup map
            file_to_idx = {fname: i for i, fname in enumerate(stored_files)}
            for fname in file_names:
                if fname in file_to_idx:
                    cache_dict[fname] = dset[file_to_idx[fname]]
                    
    return cache_dict
"""

with open('src/utils/cache_loader.py', 'w') as f:
    f.write(cache_loader_code)

print("✅ Successfully created src/utils/cache_loader.py")

In [ ]:
# 5. Load cached CQCC for Full Training Set
cqcc_cache = {}
if feat_config['features']['cqcc']['use_cached']:
    from src.utils.cache_loader import load_cached_cqcc

    # Path to the full training CQCC cache file on Google Drive
    cache_file = os.path.join(feat_config['features']['cqcc']['cache_dir'], "cqcc_cache_train.h5")

    cqcc_cache = load_cached_cqcc(
        cache_file,
        [],  # Empty list loads all entries into memory mapping for fast lookup
        feat_config['features']['cqcc']['n_cqcc'],
        feat_config['features']['cqcc']['max_frames']
    )
    print(f'Loaded full CQCC cache: {len(cqcc_cache)} files')

# 6. Import feature extractors
from src.features.mfcc import extract_mfcc
from src.features.lfcc import extract_lfcc
from src.features.spectral import extract_spectral
from src.features.fusion import compute_shared_dsp, fuse_features_for_file, compute_utterance_stats

# Setup logging
from src.utils.logger import get_project_logger
logger = get_project_logger('FeatureExtraction', 'extraction.log')

In [ ]:
# ============================================================
# FILE LIST LOADER (Smoke Test or Full Protocol Mode)
# ============================================================

if SMOKE_TEST:
    import glob
    smoke_root = '/content/drive/MyDrive/DeepFakeVoiceResearch/datasets/smoke_test'
    if isinstance(feat_config.get('smoke_test'), dict):
        smoke_root = feat_config['smoke_test'].get('root', smoke_root)
    bonafide_files = sorted(glob.glob(os.path.join(smoke_root, 'bonafide', '*.flac')))
    spoof_files = sorted(glob.glob(os.path.join(smoke_root, 'spoof', '*.flac')))
    
    smoke_cfg = feat_config.get('smoke_test', {})
    n_b = smoke_cfg.get('n_bonafide', 50) if isinstance(smoke_cfg, dict) else 50
    n_s = smoke_cfg.get('n_spoof', 50) if isinstance(smoke_cfg, dict) else 50
    
    bonafide_files = bonafide_files[:n_b]
    spoof_files = spoof_files[:n_s]
    
    file_list = bonafide_files + spoof_files
    file_labels = [0] * len(bonafide_files) + [1] * len(spoof_files)  # 0=bonafide, 1=spoof
    print(f'Smoke test batch: {len(bonafide_files)} bonafide + {len(spoof_files)} spoof = {len(file_list)} total')

else:
    import os
    
    # Resolve paths correctly using your path_config YAML layout
    asv_cfg = path_config['asvspoof_2019']
    base_root = asv_cfg['root']
    audio_dir = os.path.join(base_root, asv_cfg['train_audio'])
    protocol_dir = os.path.join(base_root, asv_cfg['protocols'])
    protocol_file = os.path.join(protocol_dir, asv_cfg['train_protocol_file'])
    
    file_list = []
    file_labels = []
    
    print(f"Loading protocol from: {protocol_file}")
    with open(protocol_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                speaker_id, file_id, system_id, eval_set, label = parts
                
                fpath = os.path.join(audio_dir, f"{file_id}.flac")
                if os.path.exists(fpath):
                    file_list.append(fpath)
                    file_labels.append(0 if label == 'bonafide' else 1)

    print(f"Loaded full dataset protocol: {len(file_list)} files mapped.")

In [ ]:
import inspect
from src.features.mfcc import extract_mfcc
from src.features.fusion import compute_shared_dsp

print("--- extract_mfcc signature ---")
print(inspect.signature(extract_mfcc))
print("\n--- compute_shared_dsp signature ---")
print(inspect.signature(compute_shared_dsp))

In [ ]:
# ============================================================
# FULL DATASET EXTRACTION LOOP WITH RESUME & CHECKPOINTING
# ============================================================
import os
import time
import numpy as np
import librosa
from src.utils.hdf5_io import init_hdf5, append_chunk, read_hdf5_split

output_path = os.path.join(FEATURES_DIR, 'extracted_features.h5')
feat_dim = 318  # Dimension validated during smoke test
BATCH_SIZE = 500  # Automatically write checkpoints to disk every 500 files

# 1. Initialize HDF5 container if it doesn't exist yet
if not os.path.exists(output_path):
    init_hdf5(output_path, feat_dim)
    print(f"Created new HDF5 container at {output_path}")

# 2. Resume logic: Check which files have already been saved
processed_files = set()
if os.path.exists(output_path):
    try:
        _, _, existing_ids = read_hdf5_split(output_path, 'train')
        if existing_ids is not None and len(existing_ids) > 0:
            processed_files = set(existing_ids.astype(str))
            print(f"RESUME MODE: Found {len(processed_files)} already processed files. Skipping them.")
    except Exception as e:
        print(f"Note: Starting fresh or could not read existing split ({e})")

# Filter out already processed files
files_to_process = []
labels_to_process = []
for fpath, label in zip(file_list, file_labels):
    fname = os.path.basename(fpath)
    if fname not in processed_files:
        files_to_process.append(fpath)
        labels_to_process.append(label)

total_files = len(files_to_process)
print(f"Remaining files to process: {total_files} / {len(file_list)}")

# 3. Resolve stats list safely
stats_def = config.get('stats', ['mean', 'std'])
if isinstance(stats_def, dict):
    stats_list = stats_def.get('functions', ['mean', 'std'])
else:
    stats_list = stats_def

# 4. Main Extraction Loop with Batch Checkpointing
batch_features = []
batch_labels = []
batch_file_ids = []

start_time = time.time()

for i, fpath in enumerate(files_to_process):
    fname = os.path.basename(fpath)
    try:
        audio, sr = librosa.load(fpath, sr=config['sample_rate'])
        dsp_cache = compute_shared_dsp(audio, sr, config)
        
        file_features = {}
        file_features['mfcc'] = extract_mfcc(dsp_cache, sr, config)
        file_features['lfcc'] = extract_lfcc(dsp_cache, sr, config)
        file_features['spectral'] = extract_spectral(dsp_cache, sr, config)
        
        if cqcc_cache and fname in cqcc_cache:
            file_features['cqcc'] = cqcc_cache[fname]

        all_stats = []
        for feat_name in ['mfcc', 'lfcc', 'spectral', 'cqcc']:
            if feat_name in file_features and file_features[feat_name] is not None:
                stats = compute_utterance_stats(file_features[feat_name], stats_list)
                all_stats.append(stats)
        
        if not all_stats:
            print(f"Skipping {fname}: No features extracted.")
            continue
            
        fused_stat = np.concatenate(all_stats)
        batch_features.append(fused_stat)
        batch_labels.append(labels_to_process[i])
        batch_file_ids.append(fname)
        
        # Write checkpoint chunk to disk when batch size is reached
        if len(batch_features) >= BATCH_SIZE:
            append_chunk(
                file_path=output_path,
                group_name='train',
                features=batch_features,
                labels=batch_labels,
                file_ids=batch_file_ids,
                checkpoint_path=None
            )
            batch_features, batch_labels, batch_file_ids = [], [], []
            
    except Exception as e:
        logger.error(f'Error processing {fpath}: {str(e)}')
        print(f'Error: {fpath}: {str(e)}')

    # Real-time progress and percentage tracking
    completed = i + 1
    if completed % 50 == 0 or completed == total_files:
        elapsed = time.time() - start_time
        rate = completed / max(elapsed, 1)
        eta = (total_files - completed) / max(rate, 0.01)
        pct = (completed / total_files) * 100
        print(f"Progress: [{completed}/{total_files}] ({pct:.1f}%) | Speed: {rate:.1f} files/s | Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m")

# Flush any remaining files in the final partial batch
if batch_features:
    append_chunk(
        file_path=output_path,
        group_name='train',
        features=batch_features,
        labels=batch_labels,
        file_ids=batch_file_ids,
        checkpoint_path=None
    )

print("\n🎉 Full dataset feature extraction complete and safely saved to HDF5!")

In [ ]:
# import inspect
# from src.utils import hdf5_io
# import src.utils.hdf5_io as hdf5_module

# # Let's rewrite the append_chunk function cleanly to avoid the UnboundLocalError
# fixed_append_chunk_code = '''
# def append_chunk(file_path, group_name, features, labels, file_ids, checkpoint_path=None):
#     import h5py
#     import numpy as np
    
#     features = np.array(features)
#     labels = np.array(labels)
    
#     with h5py.File(file_path, 'a') as hf:
#         if group_name not in hf:
#             g = hf.create_group(group_name)
#             dt = h5py.special_dtype(vlen=str)
            
#             # Create resizable datasets
#             g.create_dataset('features', data=features, maxshape=(None, features.shape[1]), chunks=True)
#             g.create_dataset('labels', data=labels, maxshape=(None,), chunks=True)
#             g.create_dataset('file_ids', data=file_ids, maxshape=(None,), dtype=dt, chunks=True)
#             new_size = features.shape[0]
#         else:
#             g = hf[group_name]
#             dset_feat = g['features']
#             dset_lab = g['labels']
#             dset_ids = g['file_ids']
            
#             old_size = dset_feat.shape[0]
#             new_size = old_size + features.shape[0]
            
#             # Resize and append
#             dset_feat.resize(new_size, axis=0)
#             dset_feat[old_size:] = features
            
#             dset_lab.resize(new_size, axis=0)
#             dset_lab[old_size:] = labels
            
#             dset_ids.resize(new_size, axis=0)
#             dset_ids[old_size:] = file_ids
            
#         hf.flush()
#     print(f"[WRITE SUCCESS] Appended {features.shape[0]} items to {group_name}. New dataset size: {new_size}")
# '''

# # Dynamically update the function in the loaded module
# exec(fixed_append_chunk_code, hdf5_module.__dict__)

# print("✅ Successfully patched append_chunk function!")

In [ ]:
# # ============================================================
# # WRITE TO HDF5
# # ============================================================
# import numpy as np
# import os
# from src.utils.hdf5_io import init_hdf5, append_chunk, read_hdf5_split

# output_path = os.path.join(FEATURES_DIR, 'extracted_features.h5')

# # Initialize HDF5
# init_hdf5(output_path, feat_dim if 'feat_dim' in dir() else 318)

# # Append all chunks
# append_chunk(
#     file_path=output_path,
#     group_name='train',
#     features=extracted_features,
#     labels=extracted_labels,
#     file_ids=extracted_file_ids,
#     checkpoint_path=None
# )

# # Verify the write
# features, labels, file_ids = read_hdf5_split(output_path, 'train')
# print(f'Verification: read {len(features)} features, shape {features.shape}')
# print(f'Labels shape: {labels.shape}, unique: {np.unique(labels)}')

# if SMOKE_TEST:
#     print('\n✅ Smoke test passed! All checks complete.')
#     print('   - Feature dims match across extractors ✓')
#     print('   - Utterance stats compute without NaN ✓')
#     print('   - HDF5 round-trip works ✓')